# Tutorial 10 — Numerical and log-linear RBC decisions

**Source:** `../PS4/Problem_Set_4_Q4.py`

**In class:** PS4 Question 4(d)–(g); the numerical policy is introduced in short stages.

Run cells from top to bottom. At each pause, predict the result before running the next cell. Plots appear below their cells; the source scripts remain the full resolution reference.


## 1. State, control, and resource constraint

The state is $(K,Z)$; hours $N$, consumption $C$, and next capital $K'$ are chosen today:

$$C+K'=ZK^\alpha N^{1-\alpha}+(1-\delta)K.$$

We will compare a numerical decision rule $K'=g(K,Z)$ with a first-order rule around the steady state. Both rules see the **same** productivity path.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.interpolate import CubicSpline
import scipy.sparse as sp
import scipy.sparse.linalg as spla


## 2. Calibrate the steady state

The steady-state Euler equation gives $r_K=1/\beta-1+\delta$. With labor share and hours fixed, calculate $\bar K$, $\bar Y$, $\bar C$, and $\theta$.


In [ ]:
BETA, ALPHA, DELTA, ETA = 0.99, 0.36, 0.025, 1.0
NBAR = 0.20
RHO, SIGMA = 0.95, 0.007

rk = 1 / BETA - 1 + DELTA
KY = ALPHA / rk
KBAR = KY ** (1 / (1 - ALPHA)) * NBAR
YBAR = KBAR**ALPHA * NBAR ** (1 - ALPHA)
CBAR = YBAR - DELTA * KBAR
WBAR = (1 - ALPHA) * YBAR / NBAR
THETA = WBAR * (1 - NBAR) ** ETA / CBAR       # from theta C (1-N)^-eta = w
EPS_F = (1 - NBAR) / (ETA * NBAR)             # Frisch elasticity


In [ ]:
print(f'K={KBAR:.3f}, Y={YBAR:.3f}, C={CBAR:.3f}, theta={THETA:.3f}')


## 3. The local rule

Use hats for log deviations from the steady state. The first-order solution has

$$\hat k_{t+1}=a_k\hat k_t+a_z\hat z_t.$$

The quadratic below yields one stable and one unstable root. The stable root respects the terminal condition.


In [ ]:
MU, YK = CBAR / KBAR, YBAR / KBAR
LAM = 1 - BETA * (1 - DELTA)
A = ALPHA + 1 / EPS_F
L = LAM * (1 - ALPHA) / A
M = MU + (1 - ALPHA) * YK / A
B = 1 / BETA + (1 - ALPHA) * YK * ALPHA / A

qa, qb, qc = 1 + L, -(B * (1 + L) + 1 + M * L / EPS_F), B
disc = qb**2 - 4 * qa * qc
roots = np.array([(-qb - np.sqrt(disc)) / (2 * qa), (-qb + np.sqrt(disc)) / (2 * qa)])
stable = roots[np.abs(roots) < 1]
assert len(stable) == 1, f"expected exactly one stable root, got {roots}"
a_k = float(stable[0])
b_k = (B - a_k) / M
d_k = (ALPHA - b_k) / A


In [ ]:
print('Candidate roots:', np.round(roots, 6), '; stable a_k =', round(a_k, 6))


In [ ]:
D = YK * (1 + (1 - ALPHA) / A)
E1 = b_k - LAM * (1 - ALPHA) * d_k + LAM * (1 - ALPHA)
E2 = LAM * RHO + LAM * (1 - ALPHA) * RHO / A
E3 = (1 - RHO) - LAM * (1 - ALPHA) * RHO / A
b_z = (E1 * D - E2) / (E3 + E1 * M)
a_z = D - M * b_z
d_z = (1 - b_z) / A


In [ ]:
print(f'k_hat_next = {a_k:.5f} k_hat + {a_z:.5f} z_hat')


## 4. Discretise the productivity state

The price-style Markov approximation now represents $\log Z_{t+1}=\rho\log Z_t+\sigma\varepsilon_{t+1}$. The grid must contain even the largest thought-experiment shock, $0.20$, so it is wider than a usual three-standard-deviation grid.


In [ ]:
def tauchen(rho, sigma, n, m):
    sd = sigma / np.sqrt(1 - rho**2)
    z = np.linspace(-m * sd, m * sd, n)
    w = z[1] - z[0]
    P = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if j == 0:
                P[i, j] = norm.cdf((z[0] - rho * z[i] + w / 2) / sigma)
            elif j == n - 1:
                P[i, j] = 1 - norm.cdf((z[-1] - rho * z[i] - w / 2) / sigma)
            else:
                P[i, j] = norm.cdf((z[j] - rho * z[i] + w / 2) / sigma) - norm.cdf(
                    (z[j] - rho * z[i] - w / 2) / sigma
                )
    return z, P


In [ ]:
N_Z, Z_RANGE = 81, .22  # live setting; source uses 161 nodes
SD_Z = SIGMA/np.sqrt(1-RHO**2)
logz, P = tauchen(RHO, SIGMA, N_Z, Z_RANGE/SD_Z)
Z = np.exp(logz)
print('log-Z range:', (logz[0], logz[-1]), 'node spacing / sigma:', (logz[1]-logz[0])/SIGMA)


## 5. Put capital near its steady state

The curved grid concentrates nodes where typical shocks keep the economy while leaving room for the large-shock comparison.


In [ ]:
N_K = 120  # live setting; source uses 300
u = np.linspace(-1, 1, N_K)
K = KBAR*(1+.9*np.sign(u)*u**2)
print('Capital range:', round(K[0],3), 'to', round(K[-1],3))


## 6. Let hours determine consumption and investment

The within-period labor condition is

$$\theta C(1-N)^{-\eta}=(1-\alpha)ZK^\alpha N^{-\alpha}.$$

For a candidate $N$, it gives $C$ directly, and the resource constraint gives $K'$. This avoids a separate search over $K'$.


In [ ]:
Kg = K[:, None]
Zg = Z[None, :]
COL = np.broadcast_to(np.arange(N_Z), (N_K, N_Z))


def from_hours(N):
    """Consumption and next-period capital implied by choosing N at each (K, Z)."""
    C = (1 - ALPHA) * Zg * Kg**ALPHA * N ** (-ALPHA) * (1 - N) ** ETA / THETA
    Kp = Zg * Kg**ALPHA * N ** (1 - ALPHA) + (1 - DELTA) * Kg - C
    return C, Kp


In [ ]:
C_test, Kp_test = from_hours(np.full((N_K,N_Z), NBAR))
print('At central grid point: C =', round(C_test[N_K//2,N_Z//2],4), ', K next =', round(Kp_test[N_K//2,N_Z//2],4))


## 7. Invert the labor condition

Given a proposed $K'$, the following bisection recovers hours. It will be used to evaluate tomorrow's marginal utility under a trial policy.


In [ ]:
def hours_at(Kc, Zc, Kn, iters=55):
    """Invert the intratemporal condition for N given (K, Z, K')."""
    lo = np.full(np.shape(Kn), 1e-10)
    hi = np.full(np.shape(Kn), 1 - 1e-10)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        C = Zc * Kc**ALPHA * mid ** (1 - ALPHA) + (1 - DELTA) * Kc - Kn
        too_low = THETA * C * (1 - mid) ** (-ETA) < (1 - ALPHA) * Zc * Kc**ALPHA * mid ** (-ALPHA)
        lo = np.where(too_low, mid, lo)
        hi = np.where(too_low, hi, mid)
    return 0.5 * (lo + hi)


## 8. Time iteration of the Euler equation

Start from a guess $K'=K$. For that guess, calculate tomorrow's marginal utility and gross return at every grid point. The matrix product with $P^\top$ takes the conditional expectation. Then bisect on today's hours to solve the Euler equation, and repeat until the policy stops changing.


In [ ]:
def solve_policy(tol=1e-12, maxit=3000):
    g = np.broadcast_to(Kg, (N_K, N_Z)).copy()      # start from K' = K
    for it in range(maxit):
        # marginal utility and gross return on the grid, under the current policy
        Ng = hours_at(Kg, Zg, g)
        Cg = Zg * Kg**ALPHA * Ng ** (1 - ALPHA) + (1 - DELTA) * Kg - g
        Rg = ALPHA * Zg * Kg ** (ALPHA - 1) * Ng ** (1 - ALPHA) + 1 - DELTA
        EE = (Rg / Cg) @ P.T                         # EE[i, j] = E[R'/C' | z_j] at K' = K_i

        lo = np.full((N_K, N_Z), 1e-6)
        hi = np.full((N_K, N_Z), 1 - 1e-6)
        for _ in range(48):
            N = 0.5 * (lo + hi)
            C, Kp = from_hours(N)
            Kc = np.clip(Kp, K[0], K[-1])
            idx = np.clip(np.searchsorted(K, Kc) - 1, 0, N_K - 2)
            w = (Kc - K[idx]) / (K[idx + 1] - K[idx])
            ee = (1 - w) * EE[idx, COL] + w * EE[idx + 1, COL]
            up = 1 / C - BETA * ee < 0               # residual rises in N
            lo = np.where(up, N, lo)
            hi = np.where(up, hi, N)
        N = 0.5 * (lo + hi)
        C, Kp = from_hours(N)
        gap = np.max(np.abs(Kp - g))
        g, Npol = Kp, N
        if gap < tol:
            break
    return g, Npol, it + 1, gap


In [ ]:
policy, Npolicy, iters, gap = solve_policy()
print(f'Numerical policy converged in {iters} iterations; change {gap:.2e}')


## 9. Look locally at the two rules

Near the steady state, the numerical rule should be close to the tangent rule. This one-state slice gives a first visual check.


In [ ]:
mid = N_Z//2
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(K, policy[:,mid], label='numerical, Z=1')
ax.plot(K, KBAR*np.exp(a_k*np.log(K/KBAR)), '--', label='first order, Z=1')
ax.set(xlabel='$K$', ylabel="$K'$", xlim=(.7*KBAR, 1.3*KBAR))
ax.legend(frameon=False); plt.show()


## 10. Simulate business cycles

Interpolate the policy between grid points, simulate a sequence of shocks, and remove slow trends from each logged series. The class sample is shorter than the 20,000-period source run, so the sample moments will move slightly.


In [ ]:
def hp_filter(y, lam=1600.0):
    """Hodrick-Prescott trend by a direct sparse solve; returns the cyclical part."""
    T = len(y)
    d = np.ones(T - 2)
    Dm = sp.diags([d, -2 * d, d], [0, 1, 2], shape=(T - 2, T), format="csc")
    trend = spla.spsolve((sp.identity(T, format="csc") + lam * Dm.T @ Dm).tocsc(), y)
    return y - trend


gspl = [CubicSpline(K, policy[:, j]) for j in range(N_Z)]
nspl = [CubicSpline(K, Npolicy[:, j]) for j in range(N_Z)]


def simulate(T=20_000, burn=1_000, seed=0):
    rng = np.random.default_rng(seed)
    cum = P.cumsum(axis=1)
    j = N_Z // 2
    k = KBAR
    out = np.empty((T, 5))                            # Y, C, I, N, w
    for t in range(T):
        kn = float(gspl[j](k))
        n = float(nspl[j](k))
        y = Z[j] * k**ALPHA * n ** (1 - ALPHA)
        out[t] = (y, y + (1 - DELTA) * k - kn, kn - (1 - DELTA) * k, n,
                  (1 - ALPHA) * y / n)
        k = kn
        j = int(np.searchsorted(cum[j], rng.random()))
    return out[burn:]


In [ ]:
gspl = [CubicSpline(K, policy[:,j]) for j in range(N_Z)]
nspl = [CubicSpline(K, Npolicy[:,j]) for j in range(N_Z)]
sim = simulate(T=4000, burn=500, seed=0)
print('Simulated observations:', len(sim))


## 11. Compare moments

Consumption should be smoother than output, investment more volatile, and hours too smooth relative to the data. The single productivity shock also makes the wage too procyclical.


In [ ]:
labels = ['Y','C','I','N','w']
cycles = np.column_stack([hp_filter(np.log(sim[:,i])) for i in range(5)])
sd = cycles.std(axis=0)*100
corr = [np.corrcoef(cycles[:,i], cycles[:,0])[0,1] for i in range(5)]
data_sd = [1.79, 1.12, 8.20, 1.76, .81]
for i, label in enumerate(labels):
    print(f'{label}: model sd {sd[i]:.2f}%, data sd {data_sd[i]:.2f}%; corr(Y) {corr[i]:.2f}')


## 12. One shock path, two decision rules

Take one innovation at date zero and no later innovations: $\hat z_t=\rho^t\varepsilon_0$. Both rules start at $\bar K$. The functions differ only in how they turn the same $(K_t,Z_t)$ into $K_{t+1}$.


In [ ]:
T_PATH = 80


In [ ]:
def path_loglinear(eps0, T=T_PATH):
    zhat = eps0 * RHO ** np.arange(T + 1)
    khat = np.zeros(T + 1)
    for t in range(T):
        khat[t + 1] = a_k * khat[t] + a_z * zhat[t]
    return KBAR * np.exp(khat)


def path_numerical(eps0, T=T_PATH):
    zhat = eps0 * RHO ** np.arange(T + 1)
    k = np.empty(T + 1)
    k[0] = KBAR
    for t in range(T):
        j = int(np.clip(np.searchsorted(logz, zhat[t]) - 1, 0, N_Z - 2))
        w = (zhat[t] - logz[j]) / (logz[j + 1] - logz[j])
        k[t + 1] = (1 - w) * float(gspl[j](k[t])) + w * float(gspl[j + 1](k[t]))
    return k


In [ ]:
shock = .01
kl, kn = path_loglinear(shock), path_numerical(shock)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(100*(kl/KBAR-1), label='first order')
ax.plot(100*(kn/KBAR-1), '--', label='numerical')
ax.set(xlabel='quarters', ylabel='$K$, % from steady state')
ax.legend(frameon=False); plt.show()
print('Maximum percentage gap:', np.max(np.abs(kn-kl)/kl)*100)


## 13. Increase the shock

Compare $\varepsilon_0\in\{.01,.05,.10,.20\}$. Express each shock in units of $\sigma=.007$. If the numerical and first-order paths diverge as the shock grows, this comparison is revealing **curvature**, not the direct effect of uncertainty.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6.5))
for ax, shock in zip(axes.flat, (.01,.05,.10,.20)):
    kl, kn = path_loglinear(shock), path_numerical(shock)
    gap = np.max(np.abs(kn-kl)/kl)*100
    ax.plot(100*(kl/KBAR-1), label='first order')
    ax.plot(100*(kn/KBAR-1), '--', label='numerical')
    ax.set(title=f'shock={shock:.2f} ({shock/SIGMA:.1f} sd), gap={gap:.3f}%', xlabel='quarters')
    print(f'shock {shock:.2f}: {shock/SIGMA:.1f} sd; max gap {gap:.4f}%')
axes[0,0].legend(frameon=False); plt.tight_layout(); plt.show()


## 14. What does the approximation leave out?

First order replaces a curved policy with its tangent near the steady state. It is also certainty equivalent: its coefficients do not depend on $\sigma$. Our path comparison mainly tests the first loss; detecting risk effects such as precautionary saving requires a higher-order solution. For shocks of one or two standard deviations, decide whether the local rule is adequate for describing the business-cycle moments considered here.
